In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

https://www.kaggle.com/datasets/algozee/financial-transaction-fraud-dataset?resource=download 

In [2]:
df = pd.read_csv(r"C:\Users\louis\Downloads\R&D PROJECTS\data\FraudShield_Banking_Data.csv")

In [3]:
df.info

<bound method DataFrame.info of        Transaction_ID  Customer_ID  Transaction_Amount (in Million)  \
0            431438.0      24239.0                              6.0   
1            902451.0      77250.0                              9.0   
2            223410.0      34294.0                              3.0   
3            145626.0      92041.0                              1.0   
4            414637.0      71578.0                              1.0   
...               ...          ...                              ...   
49995        805949.0      87401.0                              4.0   
49996        168493.0      65425.0                              4.0   
49997        996197.0      14318.0                              9.0   
49998        444042.0      55757.0                              1.0   
49999        121538.0      57274.0                              7.0   

      Transaction_Time Transaction_Date Transaction_Type  Merchant_ID  \
0                10:54       2025-03-08   

In [4]:
df.head()

,Transaction_ID,Customer_ID,Transaction_Amount (in Million),Transaction_Time,Transaction_Date,Transaction_Type,Merchant_ID,Merchant_Category,Transaction_Location,Customer_Home_Location,...,Daily_Transaction_Count,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Is_International_Transaction,Is_New_Merchant,Failed_Transaction_Count,Unusual_Time_Transaction,Previous_Fraud_Count,Fraud_Label
0,431438.0,24239.0,6.0,10:54,2025-03-08,POS,97028.0,ATM,Singapore,Lahore,...,4.0,17.0,2.0,4.0,Yes,Yes,0.0,No,1.0,Normal
1,902451.0,77250.0,9.0,19:23,2025-01-17,ATM,27515.0,ATM,Singapore,Lahore,...,4.0,9.0,5.0,8.0,Yes,Yes,1.0,No,1.0,Normal
2,223410.0,34294.0,3.0,10:20,2025-04-30,POS,13810.0,Electronics,Faisalabad,Faisalabad,...,5.0,18.0,5.0,8.0,Yes,No,0.0,Yes,1.0,Normal
3,145626.0,92041.0,1.0,14:11,2025-02-21,Online,10501.0,Grocery,London,Karachi,...,6.0,18.0,5.0,1.0,No,Yes,2.0,Yes,1.0,Normal
4,414637.0,71578.0,1.0,04:12,2025-04-11,Online,53569.0,Electronics,Singapore,Islamabad,...,3.0,18.0,4.0,3.0,No,Yes,1.0,No,1.0,Normal


## LOAD AND CLEAN THE DATASET

In [5]:
# convert target to binary
df['Fraud_Label'] = (df['Fraud_Label'] == 'Fraud').astype(int)


In [6]:
df['Transaction_Time'].unique()[:20]


array(['10:54', '19:23', '10:20', '14:11', '04:12', '15:34', '09:17',
       '22:17', '05:43', '18:06', '16:49', '23:50', '11:35', '00:51',
       '15:50', '03:49', '19:05', '15:41', '13:22', '00:04'], dtype=object)

In [7]:
# check the missing values in transaction_time
df['Transaction_Time'].isna().sum()

np.int64(9)

In [23]:
# replace 9 missing values with midnight
import datetime as dt

df['Transaction_Time'] = df['Transaction_Time'].fillna(dt.time(0, 0, 0))


In [29]:
type(df['Transaction_Date'].iloc[0]), type(df['Transaction_Time'].iloc[0])


(datetime.date, datetime.time)

In [27]:
# convert transaction date

df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce').dt.date


In [28]:
import datetime as dt
df['Transaction_Time'] = df['Transaction_Time'].fillna(dt.time(0, 0, 0))


In [30]:
# combine date + time

df['timestamp'] = df.apply(
    lambda row: pd.Timestamp.combine(row['Transaction_Date'], row['Transaction_Time']),
    axis=1
)



In [31]:
# check
df[['Transaction_Date', 'Transaction_Time', 'timestamp']].head()

,Transaction_Date,Transaction_Time,timestamp
0,2025-03-08,10:54:00,2025-03-08 10:54:00
1,2025-01-17,19:23:00,2025-01-17 19:23:00
2,2025-04-30,10:20:00,2025-04-30 10:20:00
3,2025-02-21,14:11:00,2025-02-21 14:11:00
4,2025-04-11,04:12:00,2025-04-11 04:12:00


In [32]:
# sort by time - improtant for fraud modelling

df = df.sort_values('timestamp').reset_index(drop=True)

In [35]:
# identify categorical columns
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols


['Transaction_Time',
 'Transaction_Date',
 'Transaction_Type',
 'Merchant_Category',
 'Transaction_Location',
 'Customer_Home_Location',
 'IP_Address',
 'Card_Type',
 'Is_International_Transaction',
 'Is_New_Merchant',
 'Unusual_Time_Transaction',
 'timestamp']

In [40]:
df['timestamp'].head()
df['timestamp'].dtype
type(df['timestamp'].iloc[0])


pandas._libs.tslibs.timestamps.Timestamp

In [41]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

In [42]:
# extract useful features from timestamp

df['hour'] = df['timestamp'].dt.hour
df['dayofweek'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
df['is_night'] = df['hour'].isin([0,1,2,3,4,5]).astype(int)


In [46]:
df[['timestamp', 'hour', 'dayofweek', 'is_weekend', 'is_night']].head()


,timestamp,hour,dayofweek,is_weekend,is_night
0,NaT,NaN,NaN,0,0
1,NaT,NaN,NaN,0,0
2,NaT,NaN,NaN,0,0
3,2025-01-01 00:03:00,0.0,2.0,0,1
4,2025-01-01 00:04:00,0.0,2.0,0,1


In [44]:
# check which rows are broken 

df[df['timestamp'].isna()][['Transaction_Date', 'Transaction_Time']].head(10)


,Transaction_Date,Transaction_Time
0,NaT,02:20:00
1,NaT,03:30:00
2,NaT,14:22:00


In [47]:
# fix missing dates
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce').dt.date
df['Transaction_Date'] = df['Transaction_Date'].fillna(df['Transaction_Date'].mode()[0])


In [48]:
# rebuild time stamp
df['timestamp'] = df.apply(
    lambda row: pd.Timestamp.combine(row['Transaction_Date'], row['Transaction_Time']),
    axis=1
)


In [49]:
# reextract
df['hour'] = df['timestamp'].dt.hour
df['dayofweek'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
df['is_night'] = df['hour'].isin([0,1,2,3,4,5]).astype(int)


In [50]:
df[['timestamp', 'hour', 'dayofweek', 'is_weekend', 'is_night']].head()

,timestamp,hour,dayofweek,is_weekend,is_night
0,2025-04-24 02:20:00,2,3,0,1
1,2025-04-24 03:30:00,3,3,0,1
2,2025-04-24 14:22:00,14,3,0,0
3,2025-01-01 00:03:00,0,2,0,1
4,2025-01-01 00:04:00,0,2,0,1


Finished the timestamp pipeline... Moving onto next preprocessing

In [55]:
# drop the raw timestamp, we don't want it in the model.
df = df.drop(columns=['timestamp'])

In [56]:
# identify categorical columns

categorical_cols = [
    'Transaction_Type',
    'Merchant_Category',
    'Transaction_Location',
    'Customer_Home_Location',
    'Card_Type',
    'Is_International_Transaction',
    'Is_New_Merchant',
    'Unusual_Time_Transaction'
]


In [59]:
df.columns

Index(['Transaction_ID', 'Customer_ID', 'Transaction_Amount (in Million)',
       'Transaction_Time', 'Transaction_Date', 'Transaction_Type',
       'Merchant_ID', 'Merchant_Category', 'Transaction_Location',
       'Customer_Home_Location', 'Distance_From_Home', 'Device_ID',
       'IP_Address', 'Card_Type', 'Account_Balance (in Million)',
       'Daily_Transaction_Count', 'Weekly_Transaction_Count',
       'Avg_Transaction_Amount (in Million)',
       'Max_Transaction_Last_24h (in Million)', 'Is_International_Transaction',
       'Is_New_Merchant', 'Failed_Transaction_Count',
       'Unusual_Time_Transaction', 'Previous_Fraud_Count', 'Fraud_Label',
       'hour', 'dayofweek', 'is_weekend', 'is_night'],
      dtype='object')

In [61]:
# apply label encoding - string categories into integers
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))


In [62]:
# drop identifier features
df = df.drop(columns=[
    'Transaction_ID',
    'Customer_ID',
    'Merchant_ID',
    'Device_ID',
    'IP_Address'
], errors='ignore')


In [63]:
# train/validate/split
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Fraud_Label'])
y = df['Fraud_Label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


## TRAIN BASELINE MODEL

In [74]:
# prepare data for modelling
# seperate features and target

X = df.drop(columns=['Fraud_Label'])
y = df['Fraud_Label']


In [75]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


In [76]:
# drop these before training because we already extracted needed 
df = df.drop(columns=['Transaction_Date', 'Transaction_Time'], errors='ignore')

In [77]:
# train lightgbm model

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

model = LGBMClassifier(
    objective='binary',
    learning_rate=0.05,
    num_leaves=31,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    n_estimators=1000
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=50)
    ]
)




[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Number of positive: 1696, number of negative: 33304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001258 seconds.
You can set `force_row_wise=true` to remove the 

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [78]:
# evaluate our model

from sklearn.metrics import roc_auc_score, classification_report

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

print("AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
AUC: 0.6114965235680586
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      7137
           1       0.00      0.00      0.00       363

    accuracy                           0.95      7500
   macro avg       0.48      0.50      0.49      7500
weighted avg       0.91      0.95      0.93      7500



C:\Users\louis\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\louis\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\louis\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


AUC: 0.611
This is slightly above random (0.5), but still weak.
The model can rank frauds a bit better than chance, but not by much.